# 01 - Data Audit
Scan all videos in the Google Drive folder and generate a metadata inventory.

In [ ]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior"  # <-- SET THIS to your root
OUTPUT_ROOT = "/content/drive/MyDrive/LightningPoseTrack"

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/260608.00000009"    # Input: session video folders
DRIVE_REPORTS = f"{OUTPUT_ROOT}/reports"           # Output: inventory reports

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

Cloning into '/content/LightningPoseTrack'...
remote: Enumerating objects: 785, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 785 (delta 8), reused 11 (delta 4), pack-reused 760 (from 3)
Receiving objects: 100% (785/785), 273.25 MiB | 34.18 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/LightningPoseTrack


In [ ]:
!apt-get install -y -qq tesseract-ocr > /dev/null 2>&1
!pip install --quiet opencv-python pandas numpy pyarrow pytesseract

In [ ]:
from pathlib import Path
from src.io.video_inventory import scan_videos

# Check what's in the raw videos directory
import subprocess
result = subprocess.run(["ls", "-la", DRIVE_RAW_VIDEOS], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

total 9919171
-r-------- 1 root root 15401472 May 28 15:24 002321-4.ASF
-r-------- 1 root root 15434240 May 28 15:47 004640-4.ASF
-r-------- 1 root root 15401472 May 28 17:03 020244-4.ASF
-r-------- 1 root root 19497984 May 28 17:05 020442-4.ASF
-r-------- 1 root root 19825664 May 28 18:20 031857-4.ASF
-r-------- 1 root root 19497984 May 28 18:44 034322-4.ASF
-r-------- 1 root root 15499776 May 28 19:05 040429-4.ASF
-r-------- 1 root root 15106560 May 28 21:05 060435-4.ASF
-r-------- 1 root root 53904896 May 28 21:11 060745-4.ASF
-r-------- 1 root root 35750912 May 28 21:12 061034-2.ASF
-r-------- 1 root root 16482816 May 28 21:11 061035-3.ASF
-r-------- 1 root root 24708096 May 28 21:12 061051-1.ASF
-r-------- 1 root root 18940928 May 28 21:12 061139-3.ASF
-r-------- 1 root root 58165248 May 28 21:15 061152-4.ASF
-r-------- 1 root root 20349952 May 28 21:16 061455-2.ASF
-r-------- 1 root root 16843264 May 28 21:15 061455-3.ASF
-r-------- 1 root root 47678976 May 28 21:18 061505-1.ASF


In [ ]:
df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} video files across {df['session'].nunique()} sessions")
df.head(20)

Found 316 video files across 1 sessions


,filename,path,session,camera,fps,frame_count,width,height,duration_sec,duration_min
0,002321-4.ASF,002321-4.ASF,260529.00000002,4,10.00,602,1920,1080,60.20,1.00
1,004640-4.ASF,004640-4.ASF,260529.00000002,4,10.00,603,1920,1080,60.30,1.00
2,020244-4.ASF,020244-4.ASF,260529.00000002,4,10.00,602,1920,1080,60.20,1.00
3,020442-4.ASF,020442-4.ASF,260529.00000002,4,10.00,762,1920,1080,76.20,1.27
4,034322-4.ASF,034322-4.ASF,260529.00000002,4,10.00,763,1920,1080,76.30,1.27
5,040429-4.ASF,040429-4.ASF,260529.00000002,4,10.00,602,1920,1080,60.20,1.00
6,060435-4.ASF,060435-4.ASF,260529.00000002,4,10.00,603,1920,1080,60.30,1.00
7,060745-4.ASF,060745-4.ASF,260529.00000002,4,10.00,2262,1920,1080,226.20,3.77
8,061034-2.ASF,061034-2.ASF,260529.00000002,2,10.25,1354,1920,1080,132.10,2.20
9,061035-3.ASF,061035-3.ASF,260529.00000002,3,10.00,602,1920,1080,60.20,1.00


In [ ]:
print("=== Summary ===")
print(f"Total videos: {len(df)}")
print(f"Total duration: {df['duration_min'].sum():.1f} min")
print(f"Sessions: {sorted(df['session'].unique())}")
print(f"Cameras: {sorted(df['camera'].unique())}")
print(f"\nPer session:")
print(df.groupby('session').agg(
    videos=('filename', 'count'),
    duration_min=('duration_min', 'sum'),
    cameras=('camera', lambda x: sorted(x.unique()))
))

=== Summary ===
Total videos: 316
Total duration: 737.9 min
Sessions: ['260529.00000002']
Cameras: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Per session:
                 videos  duration_min       cameras
session                                            
260529.00000002     316        737.85  [1, 2, 3, 4]


In [ ]:
output_dir = Path(DRIVE_REPORTS)
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "video_inventory.csv"
df.to_csv(csv_path, index=False)
print(f"Saved inventory to {csv_path}")

Saved inventory to /content/drive/My Drive/LightningPoseTrack/260529.00000002/reports/video_inventory.csv
